In [4]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

working from: project


In [5]:
import yfinance as yf
import pandas as pd

data = yf.download("SPY", start="2015-01-01", end="2026-08-01", progress=False)

# Flatten the MultiIndex columns down to simple names
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

print(data.shape)
print(data.head())

(2911, 5)
Price            Close        High         Low        Open     Volume
Date                                                                 
2015-01-02  169.687820  170.885549  168.655304  170.472543  121465900
2015-01-05  166.623337  168.812266  166.317716  168.647066  169632600
2015-01-06  165.053940  167.449373  164.260962  166.928980  209151400
2015-01-07  167.110672  167.449340  165.929480  166.375521  125346700
2015-01-08  170.076096  170.290867  168.498420  168.514931  147217800


In [6]:
data.to_csv(ROOT / "data" / "raw" / "spy_raw.csv")
print(f"Saved {len(data)} rows to data/raw/spy_raw.csv")

Saved 2911 rows to data/raw/spy_raw.csv


Data Source & Validation
Source: Yahoo Finance, via the yfinance Python library (API pull, no manual download or key required)
Ticker: SPY
Date range: 2015-01-01 to 2026-08-01
Rows: 2,911
Validation: no missing values found (see check below); date range and row count consistent with expected trading days over this period

In [7]:
print("Missing values per column:")
print(data.isnull().sum())

Missing values per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [8]:
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append(str(ROOT / "src"))
from utils import save_raw_data, load_raw_data

# Save using the new env-driven function
save_raw_data(data, "spy_raw.csv")

# Reload to confirm it round-trips correctly
reloaded = load_raw_data("spy_raw.csv")
print(reloaded.head())

Saved 2911 rows to data/raw/spy_raw.csv
Loaded 2911 rows from data/raw/spy_raw.csv
                 Close        High         Low        Open     Volume
Date                                                                 
2015-01-02  169.687820  170.885549  168.655304  170.472543  121465900
2015-01-05  166.623337  168.812266  166.317716  168.647066  169632600
2015-01-06  165.053940  167.449373  164.260962  166.928980  209151400
2015-01-07  167.110672  167.449340  165.929480  166.375521  125346700
2015-01-08  170.076096  170.290867  168.498420  168.514931  147217800


In [9]:
sys.path.append(str(ROOT / "src"))
from cleaning import preprocess_pipeline

raw = load_raw_data("spy_raw.csv")
processed = preprocess_pipeline(raw)

processed.to_csv(ROOT / "data" / "processed" / "spy_processed.csv")
print(f"Saved {len(processed)} processed rows")
print(processed.head())

Loaded 2911 rows from data/raw/spy_raw.csv
Removed 0 duplicate rows
Filled 0 missing values via forward-fill
Saved 2911 processed rows
                 Close        High         Low        Open     Volume
Date                                                                 
2015-01-02  169.687820  170.885549  168.655304  170.472543  121465900
2015-01-05  166.623337  168.812266  166.317716  168.647066  169632600
2015-01-06  165.053940  167.449373  164.260962  166.928980  209151400
2015-01-07  167.110672  167.449340  165.929480  166.375521  125346700
2015-01-08  170.076096  170.290867  168.498420  168.514931  147217800


Preprocessing assumptions: Missing values are forward-filled, assuming the last known price holds until new data arrives (standard for financial time series, avoids fabricating values). Duplicate dates are dropped, keeping the first occurrence. Data is sorted chronologically to support time-series modeling in later stages.